# Parameter-Space Structure of Topological LDOS Enhancement
## Phase Diagram of Edge–Emitter Coupling in Non-Hermitian SSH Lattices

**Notebook:** 07_parameter_sweep_topological_ldos  

This notebook examines how edge-enhanced light–matter interaction depends on key
system parameters in the non-Hermitian SSH lattice. Rather than focusing on a
single parameter choice, the goal here is to identify the regions of parameter
space where LDOS enhancement at the edge is present and where it breaks down.

By systematically varying the coupling ratio and the strength of
non-Hermiticity, we construct a phase-diagram-like view of edge–emitter
coupling. This allows us to distinguish robust regimes of enhancement from
fine-tuned or fragile behavior.

This notebook serves as the final step of the project, consolidating the results
of the previous analyses into a global picture of parameter dependence.

## Scope and Philosophy

The purpose of this notebook is to assess whether the edge-enhanced
light–matter interaction observed earlier is robust or restricted to
fine-tuned parameter choices. To do this, a controlled parameter sweep is
performed.

Only physically meaningful, dimensionless parameters are varied, namely the
coupling ratio \( t_2 / t_1 \) and the non-Hermiticity strength \( \gamma / t_1 \).
The analysis is based on a single observable, the LDOS enhancement factor,
which provides a clear measure of edge versus bulk response.

No new lattice models are introduced in this notebook, and no additional
physical mechanisms are assumed. The focus is strictly on mapping the behavior
of the existing model across parameter space.


In [1]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

In [2]:
def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

CONFIG_DIR = Path("..") / "config"

global_config = load_yaml(CONFIG_DIR / "global_config.yaml")
lattice_config = load_yaml(CONFIG_DIR / "lattice_params.yaml")
plotting_config = load_yaml(CONFIG_DIR / "plotting_params.yaml")

NOTEBOOK_KEY = "07_parameter_sweep_topological_ldos"
assert NOTEBOOK_KEY in plotting_config["notebook_figures"]

## Model Recap

The analysis is based on the finite non-Hermitian SSH lattice Hamiltonian used
consistently throughout the project. The lattice consists of two sites per unit
cell with alternating nearest-neighbor couplings \( t_1 \) and \( t_2 \). Gain
and loss are introduced through imaginary on-site potentials.

The real-space Hamiltonian can be written schematically as

$$
H = \sum_n \Big[
t_1 \, |n,A\rangle\langle n,B|
+ t_2 \, |n+1,A\rangle\langle n,B|
+ \text{h.c.}
\Big]
+ \sum_n \Big(
+i\gamma \, |n,A\rangle\langle n,A|
- i\gamma \, |n,B\rangle\langle n,B|
\Big).
$$


In [3]:
def build_real_space_ssh(num_cells, t1, t2, gamma):
    dim = 2 * num_cells
    H = np.zeros((dim, dim), dtype=np.complex128)

    for n in range(num_cells):
        a = 2 * n
        b = 2 * n + 1

        H[a, a] = 1j * gamma
        H[b, b] = -1j * gamma

        H[a, b] = t1
        H[b, a] = t1

        if n < num_cells - 1:
            H[a + 2, b] = t2
            H[b, a + 2] = t2

    return H

## Local Density of States (LDOS)

The local density of states (LDOS) at site \( i \) is defined in terms of the
retarded Green’s function of the photonic Hamiltonian \( H \) as

$$
\rho_i(\omega) = -\frac{1}{\pi}\,\mathrm{Im}
\left\langle i \left| (\omega - H)^{-1} \right| i \right\rangle .
$$

In the weak-coupling regime, this quantity directly governs the spontaneous
emission rate of a quantum emitter placed at site \( i \). The LDOS therefore
provides a convenient link between the spectral properties of the lattice and
light–matter interaction.


In [4]:
def local_density_of_states(H, site, omega, eta=1e-2):
    dim = H.shape[0]
    G = la.inv((omega + 1j * eta) * np.eye(dim) - H)
    return -np.imag(G[site, site]) / np.pi

## Edge LDOS Enhancement Metric

To quantify the relative enhancement of light–matter interaction at the edge, a
single scalar observable is introduced. The enhancement factor is defined as the
ratio of the maximum LDOS at an edge site to that at a bulk site,

$$
\mathcal{E} =
\frac{\max_{\omega}\,\rho_{\mathrm{edge}}(\omega)}
     {\max_{\omega}\,\rho_{\mathrm{bulk}}(\omega)} .
$$

By construction, \( \mathcal{E} \approx 1 \) indicates no significant difference
between edge and bulk response, while values \( \mathcal{E} > 1 \) signal
enhanced emission associated with edge-localized photonic modes.

In [5]:
def ldos_enhancement(H, edge_site, bulk_site, omega_vals):
    ldos_edge = np.array([
        local_density_of_states(H, edge_site, w)
        for w in omega_vals
    ])
    ldos_bulk = np.array([
        local_density_of_states(H, bulk_site, w)
        for w in omega_vals
    ])
    return np.max(ldos_edge) / np.max(ldos_bulk)


## Parameter Grid

The parameter sweep is performed over two dimensionless quantities. The first is
the coupling ratio

$$
r = \frac{t_2}{t_1},
$$

which controls the degree of dimerization in the lattice. The second parameter
is the normalized non-Hermiticity strength,

$$
\frac{\gamma}{t_1}.
$$

Restricting the sweep to dimensionless parameters allows the results to be
interpreted independently of absolute frequency or energy scales.

In [6]:
num_cells = lattice_config["lattice"]["num_unit_cells"]

r_vals = np.linspace(0.5, 1.5, 25)        # t2 / t1
gamma_vals = np.linspace(0.0, 0.6, 25)   # gamma / t1

omega_vals = np.linspace(-3, 3, 400)

edge_site = 0
bulk_site = num_cells

## Parameter Sweep

For each point in parameter space, defined by the dimensionless pair

$$
\left( \frac{t_2}{t_1}, \, \frac{\gamma}{t_1} \right),
$$

the LDOS enhancement factor \( \mathcal{E} \) is computed using the finite
non-Hermitian SSH lattice. Repeating this procedure over the full grid produces
a systematic map of how edge-enhanced light–matter interaction varies across
parameter space.


In [ ]:
enhancement_map = np.zeros((len(gamma_vals), len(r_vals)))

for i, gamma_ratio in enumerate(gamma_vals):
    for j, r in enumerate(r_vals):
        t1 = 1.0
        t2 = r * t1
        gamma = gamma_ratio * t1

        H = build_real_space_ssh(num_cells, t1, t2, gamma)

        enhancement_map[i, j] = ldos_enhancement(
            H,
            edge_site=edge_site,
            bulk_site=bulk_site,
            omega_vals=omega_vals
        )

## Parameter-space map of edge LDOS enhancement

In [ ]:
def ensure_directory(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def save_figure(fig, notebook_key, filename, plotting_config):
    cfg = plotting_config["notebook_figures"][notebook_key]
    output_dir = Path(cfg["output_dir"])
    ensure_directory(output_dir)
    path = output_dir / f"{filename}.{plotting_config['global']['figure_format']}"
    fig.savefig(path, dpi=plotting_config["global"]["dpi"], bbox_inches="tight")
    return path

fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(
    enhancement_map,
    origin="lower",
    aspect="auto",
    extent=[r_vals[0], r_vals[-1], gamma_vals[0], gamma_vals[-1]],
    cmap="viridis"
)

ax.set_xlabel("Coupling ratio t₂ / t₁")
ax.set_ylabel("Non-Hermiticity γ / t₁")
ax.set_title("Parameter-space map of edge LDOS enhancement")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("LDOS Enhancement Factor E")

fig_path = save_figure(
    fig,
    NOTEBOOK_KEY,
    "ldos_enhancement_phase_diagram",
    plotting_config
)

plt.show()
fig_path

## Interpretation

The parameter sweep shows that strong LDOS enhancement occurs primarily in the
regime

$$
t_2 > t_1,
$$

which corresponds to the topological phase of the SSH lattice. In this region,
edge-localized photonic modes contribute significantly to the local density of
states at the boundary, leading to enhanced light–matter interaction.

For moderate values of the normalized non-Hermiticity

$$
\frac{\gamma}{t_1},
$$

the LDOS enhancement is largely preserved. This indicates that the edge response
is not immediately suppressed by the presence of gain or loss. As

$$
\gamma
$$

is increased further, spectral broadening becomes significant and the
distinction between edge and bulk LDOS is gradually reduced.

Overall, these results indicate that edge-enhanced light–matter interaction
occupies a finite and robust region of parameter space, rather than arising from
a finely tuned choice of parameters.
